# SVM metadata model

Binary classification using patient metadata.

Current hypothesis:
- H1: biopsy recommendation
- H2: malignancy among biopsied lesions

Model selection is performed exclusively on the training set using patient-grouped cross-validation.

Predictor standardization is included in the SVM pipeline to ensure that variables measured on different scales contribute appropriately to the model.

## 1. Imports and configuration 

In [ ]:
# ---------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------


import numpy as np
import pandas as pd

from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedGroupKFold,
)

from sklearn.metrics import (
    auc,
    make_scorer,
    precision_recall_curve,
)

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from skin_lesion_ai.inference.evaluation import (
    evaluate,
    save_model,
)

from skin_lesion_ai.utils.data_utils import (
    load_metadata_parquet,
    subsample_training_split,
)


# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------

HYPOTHESIS = 1  # 1: Biopsy recommendation, 2: Malignancy among biopsied lesions

RANDOM_STATE = 42
N_CV_SPLITS = 5

HYPOTHESIS_CONFIG = {
    1: {
        "target": "target_biopsy",
        "split_suffix": "h1",
        "description": "Biopsy recommendation",
    },
    2: {
        "target": "target_malignant",
        "split_suffix": "h2",
        "description": "Malignancy among biopsied lesions",
    },
}

if HYPOTHESIS not in HYPOTHESIS_CONFIG:
    raise ValueError("HYPOTHESIS must be either 1 or 2.")

TARGET = HYPOTHESIS_CONFIG[HYPOTHESIS]["target"]
SPLIT_SUFFIX = HYPOTHESIS_CONFIG[HYPOTHESIS]["split_suffix"]
HYPOTHESIS_DESCRIPTION = HYPOTHESIS_CONFIG[HYPOTHESIS]["description"]

MODEL_NAME = f"svm_metadata_h{HYPOTHESIS}"

print(f"Hypothesis {HYPOTHESIS}: {HYPOTHESIS_DESCRIPTION}")
print(f"Target: {TARGET}")
print(f"Model name: {MODEL_NAME}")

## 2. Load train, validation and test splits 

In [ ]:
df_train = load_metadata_parquet(
    stage="processed",
    filename=f"train_split_{SPLIT_SUFFIX}",
    timestamp_flag=True,
)

df_validation = load_metadata_parquet(
    stage="processed",
    filename=f"val_split_{SPLIT_SUFFIX}",
    timestamp_flag=True,
)

df_test = load_metadata_parquet(
    stage="processed",
    filename=f"test_split_{SPLIT_SUFFIX}",
    timestamp_flag=True,
)


split_summary = pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "lesions": [
            len(df_train),
            len(df_validation),
            len(df_test),
        ],
        "patients": [
            df_train["patient_id"].nunique(),
            df_validation["patient_id"].nunique(),
            df_test["patient_id"].nunique(),
        ],
        "positive_rate": [
            df_train[TARGET].mean(),
            df_validation[TARGET].mean(),
            df_test[TARGET].mean(),
        ],
    }
)

display(split_summary)

In [ ]:
# ---------------------------------------------------------------------
# Data Information
# ---------------------------------------------------------------------

df_train.info()

## 3. Define predictors and target

The model uses four clinical metadata predictors:

- sex
- age
- anatomical site
- lesion diameter

Categorical variables use the encodings generated during preprocessing.
Anatomical site is represented through one-hot encoded columns rather than the integer code, avoiding the introduction of an artificial ordinal relationship.

These variables result in eight columns used as predictors by the model.

In [ ]:
FEATURES = [
    "sex_male",
    "age_approx",
    "anatom_site__anterior_torso",
    "anatom_site__head_neck",
    "anatom_site__lower_extremity",
    "anatom_site__posterior_torso",
    "anatom_site__upper_extremity",
    "clin_size_long_diam_mm",
]

ID_COLUMN = "isic_id"
GROUP_COLUMN = "patient_id"


required_columns = {
    ID_COLUMN,
    GROUP_COLUMN,
    TARGET,
    *FEATURES,
}

for split_name, df_split in {
    "train": df_train,
    "validation": df_validation,
    "test": df_test,
}.items():
    missing_columns = required_columns.difference(df_split.columns)

    if missing_columns:
        raise KeyError(
            f"{split_name} is missing required columns: {sorted(missing_columns)}"
        )

    if df_split[FEATURES].isna().any().any():
        raise ValueError(f"Missing predictor values found in {split_name}.")

print(f"Number of model columns: {len(FEATURES)}")
print("Predictors:")
for feature in FEATURES:
    print(f"- {feature}")

## 4. Prepare modelling matrices

Predictor matrices and target vectors are created for each split.

Patient identifiers are additionally retained as grouping labels for the cross-validation procedure. It is not used as a predictor. This allows `StratifiedGroupKFold` to preserve patient independence within the training set and prevents lesions from the same patient from appearing in both training and validation folds, that is to say, it uses this information to keep all lesions from the same patient within the same fold.

In [ ]:
# Training data
X_train = df_train[FEATURES].copy()
y_train = df_train[TARGET].astype(int).copy()
groups_train = df_train[GROUP_COLUMN].copy()

# Validation data
X_validation = df_validation[FEATURES].copy()
y_validation = df_validation[TARGET].astype(int).copy()

# Test data
X_test = df_test[FEATURES].copy()
y_test = df_test[TARGET].astype(int).copy()


print("Training:")
print(f"  X: {X_train.shape}")
print(f"  y: {y_train.shape}")
print(f"  patients: {groups_train.nunique():,}")

print("\nValidation:")
print(f"  X: {X_validation.shape}")
print(f"  y: {y_validation.shape}")

print("\nTest:")
print(f"  X: {X_test.shape}")
print(f"  y: {y_test.shape}")

### Prepare training subset for hyperparameter search

The complete training set is retained for final model fitting.

Because the computational cost of SVM increases substantially with the number of observations, a deterministic patient-level subset of the training data is used exclusively for hyperparameter selection.

The subset is created from the training split only (selected without using the validation or test sets). Complete patient groups are retained and the original target prevalence is approximately preserved.

The same configuration is used for both hypotheses. When the requested subset size is greater than or equal to the available training observations, the complete training split is used automatically.

Validation and test data are not used during subset selection or hyperparameter optimization.

After hyperparameter selection, the final SVM model is retrained using the complete training set.

In [ ]:
# ---------------------------------------------------------------------
# Training subset for hyperparameter search
# ---------------------------------------------------------------------

# Maximum number of training lesions used for hyperparameter search.
# If the training split contains fewer observations, the complete
# training split is used automatically.
N_SEARCH_SAMPLES = 10000


df_train_search = subsample_training_split(
    df=df_train,
    n_samples=N_SEARCH_SAMPLES,
    target_column=TARGET,
    id_column=ID_COLUMN,
    group_column=GROUP_COLUMN,
    random_state=RANDOM_STATE,
)


# ---------------------------------------------------------------------
# Prepare modelling matrices for hyperparameter search
# ---------------------------------------------------------------------

X_train_search = df_train_search[FEATURES].copy()
y_train_search = df_train_search[TARGET].astype(int).copy()
groups_train_search = df_train_search[GROUP_COLUMN].copy()


# ---------------------------------------------------------------------
# Compare complete training set and search subset
# ---------------------------------------------------------------------

search_subset_summary = pd.DataFrame(
    {
        "dataset": [
            "full_train",
            "search_subset",
        ],
        "lesions": [
            len(df_train),
            len(df_train_search),
        ],
        "patients": [
            df_train[GROUP_COLUMN].nunique(),
            df_train_search[GROUP_COLUMN].nunique(),
        ],
        "positive_rate": [
            df_train[TARGET].mean(),
            df_train_search[TARGET].mean(),
        ],
    }
)

display(search_subset_summary)


# ---------------------------------------------------------------------
# Modelling matrices summary
# ---------------------------------------------------------------------

print("Full training set:")
print(f"  X: {X_train.shape}")
print(f"  y: {y_train.shape}")
print(f"  patients: {groups_train.nunique():,}")

print("\nHyperparameter-search subset:")
print(f"  X: {X_train_search.shape}")
print(f"  y: {y_train_search.shape}")
print(f"  patients: {groups_train_search.nunique():,}")

print("\nValidation:")
print(f"  X: {X_validation.shape}")
print(f"  y: {y_validation.shape}")

print("\nTest:")
print(f"  X: {X_test.shape}")
print(f"  y: {y_test.shape}")

## 5. Cross-validation and scoring strategy

Hyperparameter optimization is performed exclusively within the training set.

Because multiple lesions may belong to the same patient, standard stratified cross-validation would introduce patient-level leakage between folds.
Therefore, a `StratifiedGroupKFold` strategy is used, preserving class balance as far as possible while ensuring that all lesions from the same patient remain within the same fold.

The same five folds are precomputed and reused throughout all hyperparameter search stages to ensure a fair comparison between candidate models.

Candidate models are evaluated using three threshold-independent metrics:

1. **PR-AUC**, used as the primary selection criterion because of the strong class imbalance.
2. **ROC-AUC**, used as the first tie-breaker.
3. **Brier score**, used as the second tie-breaker to favor better-calibrated probability estimates.

PR-AUC is computed as the trapezoidal area under the Precision-Recall curve, using the same definition as the final evaluation pipeline.

No classification threshold is selected during cross-validation. Clinical threshold selection is performed later and exclusively on the validation set.

In [ ]:
# ---------------------------------------------------------------------
# Cross-validation
# ---------------------------------------------------------------------

cv = StratifiedGroupKFold(
    n_splits=N_CV_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

# Precompute folds so that exactly the same splits are reused during all hyperparameter search stages.
cv_splits = list(
    cv.split(
        X=X_train_search,
        y=y_train_search,
        groups=groups_train_search,
    )
)


# ---------------------------------------------------------------------
# Validate CV folds
# ---------------------------------------------------------------------

search_cv_summary = []

for fold, (train_idx, val_idx) in enumerate(cv_splits, start=1):
    train_patients = set(groups_train_search.iloc[train_idx])
    val_patients = set(groups_train_search.iloc[val_idx])

    if train_patients.intersection(val_patients):
        raise ValueError(f"Patient leakage detected in CV fold {fold}.")

    search_cv_summary.append(
        {
            "fold": fold,
            "train_lesions": len(train_idx),
            "validation_lesions": len(val_idx),
            "train_patients": len(train_patients),
            "validation_patients": len(val_patients),
            "train_positive_rate": y_train_search.iloc[train_idx].mean(),
            "validation_positive_rate": y_train_search.iloc[val_idx].mean(),
        }
    )

search_cv_summary = pd.DataFrame(search_cv_summary)

display(search_cv_summary)

In [ ]:
# ---------------------------------------------------------------------
# Scoring functions
# ---------------------------------------------------------------------


def trapezoidal_pr_auc(y_true, y_prob):
    """Compute trapezoidal area under the Precision-Recall curve."""
    precision, recall, _ = precision_recall_curve(
        y_true,
        y_prob,
    )

    return auc(recall, precision)


pr_auc_scorer = make_scorer(
    trapezoidal_pr_auc,
    response_method="predict_proba",
)


SCORING = {
    "pr_auc": pr_auc_scorer,
    "roc_auc": "roc_auc",
    "brier": "neg_brier_score",
}


def select_best_candidate(cv_results):
    """Select the best candidate using PR-AUC > ROC-AUC > Brier score."""

    results = pd.DataFrame(cv_results)

    ranked_results = results.sort_values(
        by=[
            "mean_test_pr_auc",
            "mean_test_roc_auc",
            "mean_test_brier",
        ],
        ascending=[
            False,
            False,
            False,
        ],
        na_position="last",
    )

    return int(ranked_results.index[0])

## 6. Broad hyperparameter search

A broad randomized search is first performed to explore the main SVM hyperparameters.

The search evaluates the two kernels (determines the type of decision boundary) considered in this experiment:

- `linear`, which produces a linear decision boundary.
- `rbf`, which allows a non-linear decision boundary.

The main hyperparameters explored are:

- `C`, which controls the trade-off between a wider margin and classification errors;
- `gamma`, which controls the influence of individual observations when the RBF kernel is used;
- and different strategies for handling class imbalance through `class_weight`.

Parameters related mainly to the execution of the algorithm, such as `cache_size`, `max_iter` and `tol`, are fixed rather than optimized because they control computational behaviour and convergence rather than the model complexity.

The SVM predictors are standardized inside a `Pipeline` before model fitting. This prevents the scaling parameters from being estimated using validation folds during cross-validation.

Probability estimation is enabled because the evaluation pipeline requires positive-class probabilities for PR-AUC, Brier score and threshold selection.

A randomized search is used in this stage to explore the parameter space before performing a smaller exhaustive search around the best configuration.

In [ ]:
# ---------------------------------------------------------------------
# Class imbalance
# ---------------------------------------------------------------------

n_negative = int((y_train_search == 0).sum())
n_positive = int((y_train_search == 1).sum())

class_ratio = n_negative / n_positive

class_weight_values = [
    None,
    {0: 1, 1: round(np.sqrt(class_ratio), 4)},
    # optionally, you can also try the full ratio, but it may lead to overfitting
    # {0: 1, 1: round(class_ratio, 4)},
    "balanced",
]

print(f"Negative lesions: {n_negative:,}")
print(f"Positive lesions: {n_positive:,}")
print(f"Negative / positive ratio: {class_ratio:.3f}")
print("class_weight candidates:")

for value in class_weight_values:
    print(f"- {value}")

### Class imbalance

The training data presents a strong class imbalance. Therefore, different class-weighting strategies are included in the hyperparameter search.

- `None`, with equal weight for both classes;
- an intermediate positive-class weight based on the square root of the negative-to-positive ratio;
- `"balanced"`, which automatically assigns weights inversely proportional to class frequencies;
- the full negative-to-positive ratio (optional).

The purpose is to allow cross-validation to determine whether and how much class weighting improves the model's ability to identify the minority class.

### SVM pipeline

SVM is sensitive to the scale of the predictors, especially when using the RBF kernel. For this reason, the predictors are standardized before training.

The `StandardScaler` is included inside a `Pipeline` with the SVM model. This ensures that the scaling parameters are calculated independently within each training fold during cross-validation and avoids data leakage.

The same pipeline is used during hyperparameter optimization and final model fitting.

In [ ]:
# ---------------------------------------------------------------------
# Base SVM classifier
# ---------------------------------------------------------------------

svm_model = Pipeline(
    [
        ("scaler", StandardScaler()),
        (
            "svm",
            SVC(
                probability=True,
                random_state=RANDOM_STATE,
                cache_size=1024,
                max_iter=-1,
                tol=1e-3,
            ),
        ),
    ]
)


# ---------------------------------------------------------------------
# Broad search space
# ---------------------------------------------------------------------

broad_param_space = [
    {
        "svm__kernel": ["linear"],
        # "svm__C": np.logspace(-3, 3, 13),
        "svm__C": [0.01, 0.1, 0.5, 1, 5, 10, 50, 100],
        "svm__class_weight": class_weight_values,
    },
    {
        "svm__kernel": ["rbf"],
        # "svm__C": np.logspace(-3, 3, 13),
        # "svm__gamma": ["scale", "auto"] + list(np.logspace(-4, 1, 10)),
        "svm__C": [0.01, 0.1, 0.5, 1, 5, 10, 50, 100],
        "svm__gamma": ["scale", "auto", 0.01, 0.1, 0.5, 1, 5],
        "svm__class_weight": class_weight_values,
    },
]

In [ ]:
# ---------------------------------------------------------------------
# Broad randomized search
# ---------------------------------------------------------------------

N_RANDOM_ITER = 10

broad_search = RandomizedSearchCV(
    estimator=svm_model,
    param_distributions=broad_param_space,
    n_iter=N_RANDOM_ITER,
    scoring=SCORING,
    refit=select_best_candidate,
    cv=cv_splits,
    random_state=RANDOM_STATE,
    n_jobs=2,  # test with -1 y 2
    pre_dispatch="n_jobs",
    verbose=2,  # time-consuming, so we want to see progress
    return_train_score=False,
    error_score="raise",
)

broad_search.fit(
    X_train_search,
    y_train_search,
)

In [ ]:
# ---------------------------------------------------------------------
# Top broad-search candidates: show the best kernel and estimator
# ---------------------------------------------------------------------

broad_results = pd.DataFrame(broad_search.cv_results_)

display(
    broad_results[
        [
            "param_svm__kernel",
            "mean_test_pr_auc",
            "mean_test_roc_auc",
            "mean_test_brier",
            "mean_fit_time",
        ]
    ]
    .sort_values(
        by=[
            "mean_test_pr_auc",
            "mean_test_roc_auc",
            "mean_test_brier",
        ],
        ascending=[
            False,
            False,
            False,
        ],
        na_position="last",
    )
    .head(10)
)

## 7. Refined hyperparameter search

The best configuration from the broad randomized search is used to define a smaller local search space.

The refined search focuses on the main SVM parameters controlling the decision boundary:

- `C`;
- `kernel`;
- `gamma` for the RBF kernel;
- `class_weight`.

The kernel selected during the broad search is retained for the refined stage. Parameters related to the computational configuration of the SVM are kept fixed.

The broad-search optimum is explicitly included in the refined grid.

In [ ]:
# ---------------------------------------------------------------------
# Best broad-search configuration
# ---------------------------------------------------------------------

broad_best_params = broad_search.best_params_

broad_results = pd.DataFrame(broad_search.cv_results_)
broad_best_row = broad_results.loc[broad_search.best_index_]

print("Broad search best CV performance:")
print(
    f"PR-AUC: "
    f"{broad_best_row['mean_test_pr_auc']:.5f} "
    f"± {broad_best_row['std_test_pr_auc']:.5f}"
)
print(
    f"ROC-AUC: "
    f"{broad_best_row['mean_test_roc_auc']:.5f} "
    f"± {broad_best_row['std_test_roc_auc']:.5f}"
)
print(f"Brier score: {-broad_best_row['mean_test_brier']:.5f}")

print("\nBest broad-search parameters:")
display(broad_best_params)

In [ ]:
# ---------------------------------------------------------------------
# Build refined grid around broad-search optimum
# ---------------------------------------------------------------------

best_C = broad_best_params["svm__C"]
best_kernel = broad_best_params["svm__kernel"]
# best_class_weight = broad_best_params["svm__class_weight"]
if best_kernel == "rbf":
    best_gamma = broad_best_params["svm__gamma"]

if best_kernel == "linear":
    refined_param_grid = {
        "svm__kernel": ["linear"],
        "svm__C": sorted(
            {
                max(0.001, best_C / 3),
                best_C,
                best_C * 3,
            }
        ),
        "svm__class_weight": class_weight_values,  # it could be best_class_weight
    }

# if best_kernel == "rbf":
else:
    gamma_values = (
        ["scale", "auto"]
        if best_gamma in ["scale", "auto"]
        else [
            best_gamma / 3,
            best_gamma,
            best_gamma * 3,
        ]
    )

    refined_param_grid = {
        "svm__kernel": ["rbf"],
        "svm__C": sorted(
            {
                max(0.001, best_C / 3),
                best_C,
                best_C * 3,
            }
        ),
        "svm__gamma": gamma_values,
        "svm__class_weight": class_weight_values,  # it could be best_class_weight
    }

refined_param_grid

In [ ]:
# ---------------------------------------------------------------------
# Fix broad-search sampling and regularization parameters
# ---------------------------------------------------------------------

refined_svm_model = Pipeline(
    [
        ("scaler", StandardScaler()),
        (
            "svm",
            SVC(
                probability=True,
                random_state=RANDOM_STATE,
                cache_size=1024,
                max_iter=-1,
                tol=1e-3,
                # there are no sampling and regularization parameters to fix for SVM
            ),
        ),
    ]
)

In [ ]:
# ---------------------------------------------------------------------
# Refined exhaustive search
# ---------------------------------------------------------------------

refined_search = GridSearchCV(
    estimator=refined_svm_model,
    param_grid=refined_param_grid,
    scoring=SCORING,
    refit=select_best_candidate,
    cv=cv_splits,
    n_jobs=2,  # test with -1 y 2
    pre_dispatch="n_jobs",
    verbose=2,  # time-consuming, so we want to see progress
    return_train_score=False,
    error_score="raise",
)

refined_search.fit(
    X_train_search,
    y_train_search,
)

## 8. Cross-validation results and final model

The best configurations from the broad and refined searches are compared using their mean cross-validated PR-AUC, ROC-AUC and Brier score.

The final configuration is selected using the predefined hierarchy:

**PR-AUC → ROC-AUC → Brier score.**

**Hyperparameter selection** is performed exclusively on the training subset used for model selection, while the validation and test sets remain completely isolated.

Once the best hyperparameters have been identified, a new SVM pipeline is created using the selected configuration and retrained on the complete training set.

This final refit uses all available training observations and is performed only after hyperparameter selection. The validation set is therefore not used during model fitting or hyperparameter optimization.

In [ ]:
# ---------------------------------------------------------------------
# Collect CV results
# ---------------------------------------------------------------------

broad_cv_results = pd.DataFrame(broad_search.cv_results_)

refined_cv_results = pd.DataFrame(refined_search.cv_results_)


def get_best_search_result(search, search_name):
    """Return the selected CV result from one search stage."""

    results = pd.DataFrame(search.cv_results_)
    row = results.loc[search.best_index_]

    return {
        "search": search_name,
        "pr_auc": row["mean_test_pr_auc"],
        "pr_auc_std": row["std_test_pr_auc"],
        "roc_auc": row["mean_test_roc_auc"],
        "roc_auc_std": row["std_test_roc_auc"],
        "brier_score": -row["mean_test_brier"],
        "brier_score_std": row["std_test_brier"],
    }


search_comparison = pd.DataFrame(
    [
        get_best_search_result(
            broad_search,
            "Broad randomized search",
        ),
        get_best_search_result(
            refined_search,
            "Refined grid search",
        ),
    ]
)

display(search_comparison)

In [ ]:
# ---------------------------------------------------------------------
# Top refined-search candidates
# ---------------------------------------------------------------------

top_refined_models = refined_cv_results.copy()

top_refined_models["mean_brier_score"] = -top_refined_models["mean_test_brier"]

top_refined_models = (
    top_refined_models.sort_values(
        by=[
            "mean_test_pr_auc",
            "mean_test_roc_auc",
            "mean_test_brier",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    )[
        [
            "params",
            "mean_test_pr_auc",
            "std_test_pr_auc",
            "mean_test_roc_auc",
            "std_test_roc_auc",
            "mean_brier_score",
            "std_test_brier",
            "mean_fit_time",
        ]
    ]
    .head(10)
    .reset_index(drop=True)
)

display(top_refined_models)

In [ ]:
# ---------------------------------------------------------------------
# Final model:
# For reproducibility, we will use the best model from the refined search
# ---------------------------------------------------------------------

# Best hyperparameters selected during the refined search
final_model_params = refined_search.best_params_

print("Final selected hyperparameters:")
display(final_model_params)

print(
    f"Model from the refined search refitted on {len(X_train_search):,} training lesions."
)

### Training the final model with the best parameters from the refined search

In [ ]:
# ---------------------------------------------------------------------
# Clone the final model for reproducibility and fit it on the entire training set.
# ---------------------------------------------------------------------

from sklearn.base import clone

if len(X_train_search) < len(X_train):
    # Clone the selected pipeline and refit it on the complete training set
    final_model = clone(refined_search.best_estimator_)

    final_model.fit(
        X_train,
        y_train,
    )

    print(f"Final model refitted on {len(X_train):,} training lesions.")
else:
    final_model = refined_search.best_estimator_
    print(f"Final model already fitted on {len(X_train):,} training lesions.")

In [ ]:
# ---------------------------------------------------------------------
# Final model parameters: Just for the SVM parameters
# ---------------------------------------------------------------------

final_svm_params = final_model.named_steps["svm"].get_params()

final_model_params = {
    key: final_svm_params[key]
    for key in [
        "C",
        "kernel",
        "gamma",
        "class_weight",
        "probability",
        "cache_size",
        "max_iter",
        "tol",
    ]
}

print("Final selected SVM parameters:")
display(final_model_params)

print(
    f"Final model automatically refitted on {len(X_train_search):,} training lesions."
)

## 9. Save final model

The best SVM pipeline selected through cross-validation is saved before evaluating its performance on the validation set.

The complete pipeline is saved, including the `StandardScaler` and the SVM classifier, so that the same preprocessing and model configuration can be reused for future predictions without additional fitting.

---

The final SVM pipeline, selected through cross-validation and refitted on the complete training set, is saved before evaluating its performance on the validation set.

The complete pipeline is saved, including the `StandardScaler` and the SVM classifier, so that the same preprocessing and model configuration can be reused for future predictions without additional fitting.

In [ ]:
# ---------------------------------------------------------------------
# Save final model
# ---------------------------------------------------------------------

model_directory = save_model(
    model=final_model,
    model_name=MODEL_NAME,
    model_type=1,
)

print(f"Model directory: {model_directory}")

## 10. Generate prediction probabilities

Positive-class probabilities are generated for the training and validation sets using the selected SVM pipeline.

The pipeline applies the previously fitted standardization before generating the SVM probability estimates. The resulting probabilities are stored together with the lesion identifiers and are used as input for the evaluation pipeline.

---

Positive-class probabilities are generated for the complete training set and the validation set using the final SVM pipeline.

The final pipeline includes the fitted `StandardScaler` and SVM classifier. The same preprocessing and model are therefore applied consistently to both datasets.

The resulting probabilities are stored together with the lesion identifiers and are used as input for the evaluation pipeline.

In [ ]:
# ---------------------------------------------------------------------
# Generate prediction probabilities
# ---------------------------------------------------------------------

train_probabilities = final_model.predict_proba(X_train)[:, 1]

validation_probabilities = final_model.predict_proba(X_validation)[:, 1]


df_train_predictions = pd.DataFrame(
    {
        "isic_id": df_train["isic_id"].to_numpy(),
        "probability": train_probabilities,
    }
)

df_validation_predictions = pd.DataFrame(
    {
        "isic_id": df_validation["isic_id"].to_numpy(),
        "probability": validation_probabilities,
    }
)

display(df_train_predictions.head())
display(df_validation_predictions.head())

## 11. Train and validation evaluation

Threshold-independent discrimination and calibration metrics are calculated for both the training and validation sets. The clinical classification threshold is selected exclusively from the validation set by maximizing specificity while maintaining the predefined minimum sensitivity.

The selected threshold is subsequently applied unchanged to both training and validation data.

The evaluation is performed using the positive-class probabilities generated by the selected SVM pipeline.

In [ ]:
# ---------------------------------------------------------------------
# Evaluate final model: Train and validation evaluation
# ---------------------------------------------------------------------

results = evaluate(
    df_train_predictions=df_train_predictions,
    df_validation_predictions=df_validation_predictions,
    df_train=df_train,
    df_validation=df_validation,
    hypothesis=HYPOTHESIS,
    model_directory=model_directory,
    target_sensitivity=0.95,
)

display(results["summary"])

In [ ]:
from IPython.display import Image, Markdown, display

for figure_name, figure_path in results["figure_paths"].items():
    display(Markdown(f"### {figure_name.replace('_', ' ').title()}"))
    display(Image(filename=str(figure_path)))